In [9]:
import chess
import random
import json
from tqdm import tqdm

GAMES_TO_SIMULATE = 5000
MAX_MOVES_PER_GAME = 80
PARAPHRASES_PER_MOVE = 6
OUTPUT_FILE = "synthetic_chess_dataset.jsonl"

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [02:11<00:00, 38.08it/s]

Dataset generation complete.


In [ ]:
def piece_name(piece):
    names = {
        chess.PAWN: "pawn",
        chess.KNIGHT: "knight",
        chess.BISHOP: "bishop",
        chess.ROOK: "rook",
        chess.QUEEN: "queen",
        chess.KING: "king",
    }
    return names[piece.piece_type]

def square_name(square):
    return chess.square_name(square)

def generate_templates(board, move, san):
    piece = board.piece_at(move.from_square)
    target = square_name(move.to_square)

    is_capture = board.is_capture(move)
    is_castle = board.is_castling(move)
    is_promotion = move.promotion is not None

    templates = []

    if is_castle:
        if chess.square_file(move.to_square) == 6:
            templates += [
                "Castle kingside",
                "Perform kingside castling",
                "King castles short",
            ]
        else:
            templates += [
                "Castle queenside",
                "Perform queenside castling",
                "King castles long",
            ]
        return templates

    if is_promotion:
        promo_piece = chess.piece_name(move.promotion)
        templates += [
            f"Promote pawn to {promo_piece} on {target}",
            f"Advance pawn to {target} and promote to {promo_piece}",
            f"Move pawn to {target} and make it a {promo_piece}",
        ]
        return templates

    p_name = piece_name(piece)

    if is_capture:
        templates += [
            f"{p_name.capitalize()} captures on {target}",
            f"Capture the piece on {target} with the {p_name}",
            f"Take on {target} using the {p_name}",
            f"Use the {p_name} to capture on {target}",
        ]
    elif piece.piece_type == chess.PAWN:
        templates += [
            f"Advance pawn to {target}",
            f"Push the pawn to {target}",
            f"Move pawn to {target}",
        ]
    else:
        templates += [
            f"Move the {p_name} to {target}",
            f"Play {p_name} to {target}",
            f"Develop the {p_name} to {target}",
            f"{p_name.capitalize()} goes to {target}",
        ]

    templates.append(f"Play {san}")

    return templates

def create_training_entry(fen, instruction, san):
    return {
        "fen": fen,
        "instruction": instruction,
        "san": san
    }

def simulate_game():
    board = chess.Board()
    examples = []

    for _ in range(MAX_MOVES_PER_GAME):
        if board.is_game_over():
            break

        fen = board.fen()

        legal_moves = list(board.legal_moves)

        move = random.choice(legal_moves)
        san = board.san(move)

        templates = generate_templates(board, move, san)

        chosen_templates = random.sample(
            templates,
            min(len(templates), PARAPHRASES_PER_MOVE)
        )

        for instruction in chosen_templates:
            examples.append(
                create_training_entry(fen, instruction, san)
            )

        board.push(move)

    return examples

def generate_dataset():
    with open(OUTPUT_FILE, "w") as f:
        for _ in tqdm(range(GAMES_TO_SIMULATE)):
            game_examples = simulate_game()
            
            for ex in game_examples:
                f.write(json.dumps(ex) + "\n")

In [ ]:
if __name__ == "__main__":
    generate_dataset()
    print("Dataset generation complete.")

In [2]:
import chess
import numpy as np
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

MODEL_NAME = "t5-small"
DATA_FILE = "synthetic_chess_dataset.jsonl"

MAX_INPUT_LENGTH = 256
MAX_OUTPUT_LENGTH = 16

Error importing huggingface_hub.hf_file_system: module 'fsspec' has no attribute 'spec'
Error importing huggingface_hub.hf_file_system: module 'fsspec' has no attribute 'spec'


ImportError: cannot import name 'HfFileSystem' from 'huggingface_hub' (C:\Users\ChrisCyr\OneDrive - personalmicrosoftsoftware.uci.edu\Documents\School\Year 5\Q2 - Winter\cs175\Project\Lib\site-packages\huggingface_hub\__init__.py)

In [ ]:
dataset = load_dataset("json", data_files=DATA_FILE)["train"]

dataset = dataset.train_test_split(test_size=0.05)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

In [ ]:
def format_input(example):
    return (
        f"translate chess move: "
        f"fen: {example['fen']} "
        f"instruction: {example['instruction']}"
    )

def preprocess(example):

    input_text = format_input(example)

    model_inputs = tokenizer(
        input_text,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        example["san"],
        max_length=MAX_OUTPUT_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


train_dataset = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(preprocess, remove_columns=eval_dataset.column_names)


data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

def compute_metrics(eval_preds):

    preds, labels = eval_preds

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    exact_matches = []
    legal_moves = []

    for pred, label in zip(decoded_preds, decoded_labels):

        pred = pred.strip()
        label = label.strip()

        exact_matches.append(pred == label)

        # Legality check
        try:
            # We cannot access FEN directly here, so legality
            # is approximated by parsing SAN format.
            chess.Board().parse_san(pred)
            legal_moves.append(True)
        except:
            legal_moves.append(False)

    return {
        "exact_match": np.mean(exact_matches),
        "legal_format": np.mean(legal_moves),
    }

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./chess_t5_model",

    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,

    predict_with_generate=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",

    logging_steps=200,
    save_total_limit=2,

    fp16=True
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


trainer.train()

trainer.save_model("./chess_t5_model")
tokenizer.save_pretrained("./chess_t5_model")